# 11장. 이야기 데이터와 어텐션 헤드

이야기 데이터를 불러와 번호로 바꾸고, 어텐션 헤드 하나를 다섯 단계(점수 → 나누기 → 가리기 → 비율 → 모으기)로 직접 계산합니다. 마지막에 패키지의 Head 부품과 결과가 같은지 확인합니다.

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0)

## 실습 11-1 이야기 데이터 불러오기

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import load_stories, Vocab, encode_texts

stories = load_stories(200)
print("이야기 수:", len(stories))
print(stories[0][:100])

## 실습 11-2 글자 사전으로 번호 바꾸기

In [ ]:
vocab = Vocab.build()
data = encode_texts(stories, vocab)
print("사전 크기:", len(vocab))
print("글자 수:", len(data))
print("처음 10글자:", data[:10])
print("되돌리기:", vocab.decode(data[:10]))

## 실습 11-3 글자를 벡터로 바꾸기

In [ ]:
from modu_llm import GPTConfig, Embeddings

torch.set_grad_enabled(False)      # 이 장에서는 학습하지 않으므로 기울기 계산을 끕니다
torch.manual_seed(42)
cfg = GPTConfig(n_embd=16)
emb = Embeddings(cfg)

sentence = "토끼가 숲에 가요"
idx = torch.tensor(vocab.encode(sentence))
x = emb(idx)
print("글자 번호:", idx.tolist())
print("x 모양:", x.shape)

## 실습 11-4 질문 쪽지, 명찰, 메모 만들기

In [ ]:
head_size = 8
query = nn.Linear(16, head_size, bias=False)   # 질문 쪽지를 만드는 부품
key = nn.Linear(16, head_size, bias=False)     # 명찰을 만드는 부품
value = nn.Linear(16, head_size, bias=False)   # 메모를 만드는 부품

q = query(x)
k = key(x)
v = value(x)
print("q:", q.shape)
print("k:", k.shape)
print("v:", v.shape)

## 실습 11-5 ① 점수와 ② 나누기

In [ ]:
scores = q @ k.T                      # ① 점수: 질문 쪽지와 명찰의 내적
scores = scores / head_size ** 0.5    # ② 나누기
print("점수 표 모양:", scores.shape)
print("'가'의 줄:", scores[2].round(decimals=2))

## 실습 11-6 ③ 가리기와 ④ 비율

In [ ]:
from modu_llm import show_weights

T = len(sentence)
mask = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(mask == 0, float("-inf"))   # ③ 가리기
weights = F.softmax(scores, dim=-1)                       # ④ 비율

labels = list(sentence.replace(" ", "␣"))
print("줄마다 합:", weights.sum(dim=-1).round(decimals=2))
show_weights(weights, labels)

## 실습 11-7 ⑤ 모으기

In [ ]:
out = weights @ v                      # ⑤ 모으기
print("out 모양:", out.shape)
print("'토'의 결과와 '토'의 메모가 같은가요?", torch.allclose(out[0], v[0]))

## 실습 11-8 패키지 부품과 결과 비교하기

In [ ]:
from modu_llm import Head

head = Head(cfg, head_size)
head.query = query
head.key = key
head.value = value
print("결과가 같은가요?", torch.allclose(head(x), out))
print("비율 표가 같은가요?", torch.allclose(head.weights, weights))